# Commentary Ingestion Pipeline

## Directory structure
```
data/commentary/
  raw/          ← one JSON per race: video metadata + transcript
  extracted/    ← one JSON per race: Claude tactical extraction
```

## Daily workflow
1. **Run Cell 8** — automated video search (uses YouTube API quota, ~90 races/day)
2. **Run Cell 9** — transcript fetch for everything found (no quota, space requests out)
3. **Run Cell 10** — manual override for any IP-blocked or no-video races
4. **Run Cell 12** — Claude extraction (run whenever, costs ~$0.02/race)

## Status checks
- **Cell 7** — see what's been found, fetched, failed at any time


In [31]:
import os
import re
import json
import time
import random
import datetime
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)
from rapidfuzz import fuzz
import anthropic

load_dotenv()

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
PROCESSED_DATA  = Path('../data/processed')
RAW_DIR         = Path('../data/commentary/raw')
EXTRACTED_DIR   = Path('../data/commentary/extracted')
RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

ytt     = YouTubeTranscriptApi()
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
claude  = anthropic.Anthropic()

print(f'YouTube API key: {"SET" if YOUTUBE_API_KEY else "MISSING"}')
print(f'Raw dir:         {RAW_DIR}')
print(f'Extracted dir:   {EXTRACTED_DIR}')
print(f'Raw files:       {len(list(RAW_DIR.glob("*.json")))}')
print(f'Extracted files: {len(list(EXTRACTED_DIR.glob("*.json")))}')


YouTube API key: SET
Raw dir:         ..\data\commentary\raw
Extracted dir:   ..\data\commentary\extracted
Raw files:       126
Extracted files: 0


In [32]:
CHANNEL_REGISTRY = [
    {'id': 'UCqZQlzSHbVJrwrn5XvzrzcA', 'name': 'NBC Sports',        'coverage': 'Grand Tours, Monuments, WorldTour'},
    {'id': 'UCu7phdCr-raU7OaJfEpHZww', 'name': 'GCN Racing',         'coverage': 'All WorldTour'},
    {'id': 'UCfDfvvMARk4TKcC62ALi6eA', 'name': 'TNT Sports Cycling', 'coverage': 'Grand Tours, European classics'},
    {'id': 'UCuTaETsuCOkJ0H_GAztWt0Q', 'name': 'GCN',                'coverage': 'All WorldTour'},
]
print(f'Channels: {[c["name"] for c in CHANNEL_REGISTRY]}')


Channels: ['NBC Sports', 'GCN Racing', 'TNT Sports Cycling', 'GCN']


In [ ]:
# ── Search and scoring functions ─────────────────────────

def search_channel(race_name, channel_id, race_date, stage=None, max_results=3):
    race_date = str(race_date)[:10]
    race_dt   = datetime.datetime.strptime(race_date, '%Y-%m-%d')
    query     = f'{race_name} {race_date[:4]} stage {stage} highlights' if stage \
                else f'{race_name} {race_date[:4]} highlights'
    pub_after  = (race_dt - datetime.timedelta(days=3)).strftime('%Y-%m-%dT00:00:00Z')
    pub_before = (race_dt + datetime.timedelta(days=30)).strftime('%Y-%m-%dT00:00:00Z')
    try:
        resp = youtube.search().list(
            part='snippet', channelId=channel_id, q=query,
            type='video', maxResults=max_results, order='relevance',
            publishedAfter=pub_after, publishedBefore=pub_before,
        ).execute()
        return [{
            'video_id':  item['id']['videoId'],
            'title':     item['snippet']['title'],
            'published': item['snippet']['publishedAt'],
            'url':       f"https://youtube.com/watch?v={item['id']['videoId']}",
            'channel':   next((c['name'] for c in CHANNEL_REGISTRY if c['id'] == channel_id), 'unknown'),
        } for item in resp.get('items', [])]
    except HttpError as e:
        if 'quotaExceeded' in str(e):
            print('Daily quota limit reached — resets at midnight Pacific')
            raise
        return []


def score_video(video, race_name, race_date, stage=None):
    score     = 0.0
    title     = video['title'].lower()
    race_date = str(race_date)[:10]
    race_dt   = datetime.datetime.strptime(race_date, '%Y-%m-%d')
    try:
        pub_dt = datetime.datetime.strptime(video['published'][:10], '%Y-%m-%d')
    except ValueError:
        return -999.0
    days_diff = abs((pub_dt - race_dt).days)
    if days_diff == 0:      score += 60
    elif days_diff <= 1:    score += 50
    elif days_diff <= 3:    score += 35
    elif days_diff <= 7:    score += 15
    elif days_diff > 30:    score -= 40 * abs(pub_dt.year - race_dt.year)
    name_score = fuzz.partial_ratio(race_name.lower(), title)
    score += name_score * 0.25
    if name_score < 40:     score -= 40
    if stage:
        score += 30 if any(p in title for p in [f'stage {stage}', f'stage{stage}',
                           f'étape {stage}', f'tappa {stage}']) else -10
    if str(race_dt.year) in title:  score += 15
    if 'extended' in title:         score += 10
    elif 'highlights' in title:     score += 5
    if 'nbc' in video.get('channel','').lower() and 'extended' in title: score += 5
    # skip = ['preview','interview','training','behind the scenes','beginner',
    #         'guide','how to','shorts','power data']
    skip = [
        'preview', 'interview', 'training', 'behind the scenes',
        'beginner', 'guide', 'how to', 'shorts', 'power data',
        'riders to watch', 'ones to watch', 'top 10',
        'what to watch', 'race preview', 'stage preview',
        'team time trial preparation',
    ]
    if any(k in title for k in skip):   score -= 20
    if any(k in title for k in ['femmes','feminine','women','ladies']): score -= 30
    return round(score, 2)


def find_best_video(race_name, race_date, stage=None, threshold=100.0, verbose=False):
    all_candidates = []
    for channel in CHANNEL_REGISTRY:
        if verbose: print(f'  Searching {channel["name"]}...')
        results = search_channel(race_name, channel['id'], race_date, stage)
        for v in results:
            v['match_score'] = score_video(v, race_name, race_date, stage)
        all_candidates.extend(results)
        if all_candidates:
            best = max(all_candidates, key=lambda x: x['match_score'])
            if best['match_score'] >= threshold:
                if verbose: print(f'  High-confidence match found — stopping early')
                break
        time.sleep(0.3)
    if not all_candidates:
        return {'found': False}
    best = max(all_candidates, key=lambda x: x['match_score'])
    best['found']          = True
    best['all_candidates'] = all_candidates
    return best


Search functions ready


In [ ]:
def fetch_transcript(video_id: str) -> dict:
    try:
        transcript = ytt.fetch(video_id)
        raw_text   = ' '.join([s.text for s in transcript])
        clean_text = re.sub(r'\[.*?\]', '', raw_text)
        clean_text = re.sub(r'\(.*?\)', '', clean_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        duration   = round(transcript[-1].start, 0) if transcript else 0
        return {
            'success':       True,
            'video_id':      video_id,
            'snippet_count': len(transcript),
            'raw_chars':     len(raw_text),
            'clean_chars':   len(clean_text),
            'duration_mins': round(duration / 60, 1),
            'clean_text':    clean_text,
            'preview_start': clean_text[:500],
            'preview_end':   clean_text[-500:],
            'error':         None,
        }
    except NoTranscriptFound:
        return {'success': False, 'video_id': video_id, 'error': 'no_transcript'}
    except TranscriptsDisabled:
        return {'success': False, 'video_id': video_id, 'error': 'transcripts_disabled'}
    except VideoUnavailable:
        return {'success': False, 'video_id': video_id, 'error': 'video_unavailable'}
    except Exception as e:
        return {'success': False, 'video_id': video_id, 'error': str(e)}


def save_transcript(label, race_name, race_date, stage, video_meta, transcript) -> Path:
    """Save a successfully fetched transcript to RAW_DIR."""
    safe_name = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    out_path  = RAW_DIR / f'{safe_name}.json'
    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  str(race_date)[:10],
        'stage':      stage,
        'video':      video_meta,
        'transcript': {
            'snippet_count': transcript['snippet_count'],
            'raw_chars':     transcript['raw_chars'],
            'clean_chars':   transcript['clean_chars'],
            'duration_mins': transcript['duration_mins'],
            'clean_text':    transcript['clean_text'],
            'preview_start': transcript['preview_start'],
            'preview_end':   transcript['preview_end'],
        },
        'status': 'transcript_saved',
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    return out_path


def save_no_video(label, race_name, race_date, stage, reason='no_video_found') -> Path:
    """Save a record for races where no video was found — so we skip them tomorrow."""
    safe_name = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    out_path  = RAW_DIR / f'{safe_name}.json'
    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  str(race_date)[:10],
        'stage':      stage,
        'video':      None,
        'transcript': None,
        'status':     reason,
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    return out_path

Transcript functions ready


In [35]:
def parse_race_name_and_stage(race_name_full: str):
    clean       = re.sub(r'^\d{4}\s+', '', race_name_full).strip()
    stage_match = re.search(r'Stage\s+(\d+)', clean, re.IGNORECASE)
    stage       = int(stage_match.group(1)) if stage_match else None
    race        = re.sub(r'\s*Stage\s+\d+', '', clean, flags=re.IGNORECASE).strip()
    return race, stage


def make_label(race_name, race_date, stage):
    label = f'{str(race_date)[:4]} {race_name}'
    return label + (f' Stage {stage}' if stage else '')


def make_safe_name(label):
    return re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')


# Build race index
merged_df = pd.read_csv(PROCESSED_DATA / 'merged_uci_races.csv', low_memory=False)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

race_index = (
    merged_df[['Race Name', 'Race_results', 'Date', 'Year_results', 'Stage_results']]
    .drop_duplicates('Race Name')
    .sort_values('Date')
    .reset_index(drop=True)
)

# Count current status
raw_files    = list(RAW_DIR.glob('*.json'))
saved_names  = {f.stem for f in raw_files}

statuses = {}
for path in raw_files:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', 'unknown')
    statuses[s] = statuses.get(s, 0) + 1

transcript_saved = statuses.get('transcript_saved', 0)
no_video         = statuses.get('no_video_found', 0)
ip_errors        = sum(v for k, v in statuses.items() if 'block' in k.lower() or 'ip' in k.lower())
other_errors     = sum(v for k, v in statuses.items()
                       if k not in ['transcript_saved', 'no_video_found', 'unknown']
                       and 'block' not in k.lower() and 'ip' not in k.lower())
remaining        = len(race_index) - len(raw_files)

print(f'Total races in dataset:  {len(race_index):,}')
print(f'Already processed:       {len(raw_files):,}')
print(f'  Transcripts saved:     {transcript_saved}')
print(f'  No video found:        {no_video}')
print(f'  IP/transcript errors:  {ip_errors + other_errors}')
print(f'Remaining (unprocessed): {remaining}')
print(f'Est. days at 90/day:     {remaining // 90 + 1}')


Total races in dataset:  896
Already processed:       126
  Transcripts saved:     80
  No video found:        46
  IP/transcript errors:  80
Remaining (unprocessed): 770
Est. days at 90/day:     9


In [37]:
# import json
# from pathlib import Path

# RAW_DIR = Path("../data/commentary/raw")

# test_files = [
#     "2017_vuelta_a_espana_stage_1.json",
#     "2017_vuelta_a_espana_stage_2.json",
# ]

# for fname in test_files:
#     path = RAW_DIR / fname
#     if path.exists():
#         with open(path) as f:
#             data = json.load(f)
#         print(f"File: {fname}")
#         print(f"  status:     {data.get('status')}")
#         print(f"  has video:  {data.get('video') is not None}")
#         print(f"  video_id:   {data.get('video', {}).get('video_id', 'MISSING')}")
#         print(f"  transcript: {data.get('transcript') is not None}")
#         print()
#     else:
#         print(f"File not found: {fname}")

---
## Cell 8 — Automated video search
Run daily. Uses YouTube API quota (~90 races per day).
Saves video metadata but does NOT fetch transcripts yet.
Already-processed races are skipped automatically.


In [ ]:
def run_automated_search(max_races: int = 90, verbose: bool = False) -> dict:
    """
    Search YouTube for video IDs for all unprocessed races.
    Saves a record for each race — either video metadata or no_video_found.
    Does NOT fetch transcripts — that happens separately in Cell 9.
    Safe to run daily — skips anything already in RAW_DIR.
    """
    processed = skipped = found = not_found = errors = 0
    no_video_races = []

    print(f'Starting automated search: up to {max_races} new races')
    print(f'{chr(8212)*55}')

    for i, row in race_index.iterrows():
        if processed >= max_races:
            break

        race_name, stage = parse_race_name_and_stage(row['Race Name'])
        race_date        = str(row['Date'])[:10]
        label            = make_label(race_name, race_date, stage)
        safe_name        = make_safe_name(label)
        out_path         = RAW_DIR / f'{safe_name}.json'

        if out_path.exists():
            skipped += 1
            continue

        if processed % 10 == 0:
            print(f'  [{processed:>3} new | {found} found | {not_found} no_video | {errors} errors]')

        try:
            video = find_best_video(race_name, race_date, stage, verbose=verbose)

            if video.get('found'):
                # Save video metadata — transcript fetch happens in Cell 9
                record = {
                    'label':      label,
                    'race_name':  race_name,
                    'race_date':  race_date,
                    'stage':      stage,
                    'video': {
                        'video_id':       video['video_id'],
                        'title':          video['title'],
                        'url':            video['url'],
                        'channel':        video['channel'],
                        'published':      video['published'],
                        'match_score':    video['match_score'],
                        'all_candidates': [
                            {'video_id': c['video_id'], 'title': c['title'],
                             'channel': c['channel'], 'published': c['published'],
                             'match_score': c['match_score']}
                            for c in video.get('all_candidates', [])
                        ],
                    },
                    'transcript': None,
                    'status':     'video_found',   # transcript not yet fetched
                }
                with open(out_path, 'w', encoding='utf-8') as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)
                found += 1
                if verbose:
                    print(f'  ✓ [{video["match_score"]:.0f}] {video["title"][:55]}')
            else:
                save_no_video(label, race_name, race_date, stage)
                no_video_races.append(label)
                not_found += 1

        except HttpError as e:
            if 'quotaExceeded' in str(e):
                print(f'\n⚠ Quota exhausted after {found} new videos found.')
                print(f'  Re-run tomorrow — already-saved races will be skipped.')
                break
            errors += 1
        except Exception as e:
            errors += 1
            if verbose:
                print(f'  Error: {e}')

        processed += 1
        time.sleep(0.5)

    print(f'\n{chr(8212)*55}')
    print(f'Search complete')
    print(f'  Videos found:    {found}')
    print(f'  No video:        {not_found}')
    print(f'  Skipped:         {skipped}')
    print(f'  Errors:          {errors}')

    if no_video_races:
        print(f'\n── No video found (add manually if you can find them) ──')
        for label in no_video_races:
            print(f'  {label}')

    return {'found': found, 'not_found': not_found,
            'skipped': skipped, 'errors': errors}


# ── RUN THIS DAILY ─────────────────────────────────────────
search_stats = run_automated_search(
    max_races=90,      # safe daily limit within 10,000 quota units
    verbose=True,     # set True to see each video found
)


Starting automated search: up to 90 new races
———————————————————————————————————————————————————————
  [  0 new | 0 found | 0 no_video | 0 errors]
  [ 10 new | 5 found | 5 no_video | 0 errors]
  [ 20 new | 9 found | 11 no_video | 0 errors]
Daily quota limit reached — resets at midnight Pacific

⚠ Quota exhausted after 11 new videos found.
  Re-run tomorrow — already-saved races will be skipped.

———————————————————————————————————————————————————————
Search complete
  Videos found:    11
  No video:        14
  Skipped:         96
  Errors:          0

── No video found (add manually if you can find them) ──
  2017 BinckBank Tour Stage 4
  2017 BinckBank Tour Stage 5
  2017 BinckBank Tour Stage 6
  2017 BinckBank Tour Stage 7
  2017 BinckBank Tour Stage 1
  2017 Vuelta a Espana Stage 7
  2017 Bretagne Classic - Ouest France
  2017 Vuelta a Espana Stage 11
  2017 Vuelta a Espana Stage 13
  2017 Vuelta a Espana Stage 14
  2017 Vuelta a Espana Stage 15
  2017 Vuelta a Espana Stage 17
  2

---
## Cell 9 — Fetch transcripts
Run after Cell 8. No API quota used — just HTTP requests to YouTube.
Uses generous delays to avoid IP blocking.
Safe to run multiple times per day.


In [10]:
def run_transcript_fetch(
    max_transcripts: int = 50,
    delay_seconds: float = 45.0,
) -> dict:
    """
    Fetch transcripts for all races where a video was found
    but transcript not yet downloaded.
    Uses deliberate delays to avoid YouTube IP blocking.
    No YouTube Data API calls — zero quota cost.
    Safe to run multiple times per day.
    """
    # Find all races with video_found status (need transcript)
    pending = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        if data.get('status') == 'video_found':
            pending.append((path, data))

    total = min(len(pending), max_transcripts)
    print(f'Videos pending transcript: {len(pending)}')
    print(f'Fetching up to:            {total}')
    print(f'Delay between requests:    {delay_seconds}s')
    print(f'Estimated time:            {total * delay_seconds / 60:.1f} minutes')
    print(f'{chr(8212)*55}')

    success = errors = ip_blocked = 0

    for path, data in pending[:max_transcripts]:
        label     = data['label']
        video     = data['video']
        video_id  = video['video_id']

        print(f'\nFetching: {label}')
        print(f'  {video["title"][:60]}')

        transcript = fetch_transcript(video_id)

        if transcript['success']:
            # Update the existing file with transcript data
            data['transcript'] = {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            }
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | '
                  f'{transcript["duration_mins"]:.1f} min')
            success += 1
        else:
            error_msg = transcript['error']
            print(f'  ✗ {error_msg[:80]}')

            # Detect IP blocking specifically
            is_ip_block = any(kw in error_msg.lower()
                              for kw in ['blocked', 'ip', '429', 'too many'])

            if is_ip_block:
                # Update status so we know to retry
                data['status'] = f'transcript_error:ip_blocked'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                ip_blocked += 1
                print(f'\n⚠ IP blocking detected — stopping.')
                print(f'  Saved {success} transcripts before block.')
                print(f'  These {ip_blocked} races kept status "transcript_error:ip_blocked"')
                print(f'  They will be retried next time you run this cell.')
                print(f'  Wait a few hours if blocking persists.')
                break
            else:
                # Non-IP error — save as permanent failure
                data['status'] = f'transcript_error:{error_msg[:80]}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1

        # Deliberate delay with jitter — looks human to YouTube
        actual_delay = delay_seconds + random.uniform(-5, 10)
        actual_delay = max(20, actual_delay)
        print(f'  Waiting {actual_delay:.0f}s...')
        time.sleep(actual_delay)

    # Recount ip_blocked races for retry next time
    ip_blocked_total = sum(
        1 for path in RAW_DIR.glob('*.json')
        if json.load(open(path)).get('status') == 'transcript_error:ip_blocked'
    )

    print(f'\n{chr(8212)*55}')
    print(f'Fetch complete')
    print(f'  Transcripts saved:   {success}')
    print(f'  IP blocked:          {ip_blocked}')
    print(f'  Other errors:        {errors}')
    print(f'  Still pending retry: {ip_blocked_total}')
    return {'success': success, 'ip_blocked': ip_blocked, 'errors': errors}


In [ ]:
# ── RUN THIS AFTER CELL 8 ─────────────────────────────────
# Increase max_transcripts if no IP issues — can safely do 100+ per day
# with 45s delays. Reduce delay_seconds if you want faster processing
# but increase risk of IP blocking.
fetch_stats = run_transcript_fetch(
    max_transcripts=50,
    delay_seconds=45.0,
)

---
## Cell 10 — Manual override
Use this to:
- Add a video ID you found manually for a 'no_video_found' race
- Retry an IP-blocked race immediately
- Override an incorrect automated match with a better video

Find the video on YouTube, copy the ID from the URL after `?v=`


In [11]:
def manual_add(
    video_id: str,
    label: str = None,
) -> dict:
    """
    Manually fetch a transcript by video ID.

    If label is provided, uses it directly.
    If label is None, shows you the pending list so you can
    copy-paste the exact label string.

    Usage:
        # Step 1 — see what needs manual attention
        show_manual_todo()

        # Step 2 — copy a label from that list, paste here
        manual_add("VIDEO_ID_HERE", "2017 Paris-Roubaix")
    """
    if label is None:
        print("Provide a label from show_manual_todo() output.")
        print("Example: manual_add('VIDEO_ID', '2017 Paris-Roubaix')")
        show_manual_todo()
        return {}

    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'

    # Try to find existing metadata for this label
    race_name = label
    race_date = "2017-01-01"
    stage     = None

    if out_path.exists():
        # Load whatever we already know about this race
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        race_name = existing.get('race_name', label)
        race_date = existing.get('race_date', '2017-01-01')
        stage     = existing.get('stage')
        print(f'Overwriting existing record: {existing.get("status")}')
    else:
        # Try to find it in race_index
        matches = race_index[
            race_index['Race Name'].str.contains(
                label.split(' ', 1)[-1],  # strip year prefix
                na=False, case=False
            )
        ]
        if not matches.empty:
            row       = matches.iloc[0]
            race_name, stage = parse_race_name_and_stage(row['Race Name'])
            race_date = str(row['Date'])[:10]

    print(f'Manual fetch: {label}')
    print(f'  Video ID:  {video_id}')
    print(f'  Race:      {race_name}')
    print(f'  Date:      {race_date}')
    print(f'  Stage:     {stage}')

    transcript = fetch_transcript(video_id)

    if not transcript['success']:
        print(f'  ✗ Failed: {transcript["error"]}')
        return {'success': False, 'error': transcript['error']}

    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  race_date,
        'stage':      stage,
        'video': {
            'video_id':       video_id,
            'title':          'manually provided',
            'url':            f'https://youtube.com/watch?v={video_id}',
            'channel':        'manual',
            'published':      race_date,
            'match_score':    999,
            'all_candidates': [],
        },
        'transcript': {
            'snippet_count': transcript['snippet_count'],
            'raw_chars':     transcript['raw_chars'],
            'clean_chars':   transcript['clean_chars'],
            'duration_mins': transcript['duration_mins'],
            'clean_text':    transcript['clean_text'],
            'preview_start': transcript['preview_start'],
            'preview_end':   transcript['preview_end'],
        },
        'status': 'transcript_saved',
    }

    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f'  ✓ {transcript["clean_chars"]:,} chars | '
          f'{transcript["duration_mins"]:.1f} min')
    print(f'  Saved: {out_path.name}')
    return {'success': True, 'chars': transcript['clean_chars']}


def show_manual_todo() -> None:
    """
    Show all races that need manual attention —
    no video found or IP blocked.
    Prints the exact label string to copy-paste into manual_add().
    """
    no_video  = []
    ip_blocked = []

    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        status = data.get('status', '')
        label  = data.get('label', path.stem)
        if status == 'no_video_found':
            no_video.append(label)
        elif 'ip_blocked' in status or 'blocked' in status.lower():
            ip_blocked.append(label)

    if no_video:
        print(f'\n── No video found ({len(no_video)}) ──')
        print('Search YouTube for these, then call:')
        print('manual_add("VIDEO_ID", "LABEL")\n')
        for label in no_video:
            print(f'  "{label}"')

    if ip_blocked:
        print(f'\n── IP blocked — retry these ({len(ip_blocked)}) ──')
        print('Either wait and retry, or find video manually:\n')
        for label in ip_blocked:
            print(f'  "{label}"')

    if not no_video and not ip_blocked:
        print('Nothing needs manual attention.')

In [12]:
# Step 1 — see what needs manual attention
show_manual_todo()


── No video found (41) ──
Search YouTube for these, then call:
manual_add("VIDEO_ID", "LABEL")

  "2017 BinckBank Tour Stage 1"
  "2017 BinckBank Tour Stage 2"
  "2017 BinckBank Tour Stage 3"
  "2017 BinckBank Tour Stage 4"
  "2017 BinckBank Tour Stage 5"
  "2017 BinckBank Tour Stage 6"
  "2017 BinckBank Tour Stage 7"
  "2017 Bretagne Classic - Ouest France"
  "2017 Eschborn-Frankfurt 'Rund um den Finanzplatz'"
  "2017 Giro d'Italia Stage 14"
  "2017 Giro d'Italia Stage 18"
  "2017 Giro d'Italia Stage 19"
  "2017 La Fleche Wallonne"
  "2017 Liege - Bastogne - Liege"
  "2017 Paris-Nice Stage 8"
  "2017 Tirreno-Adriatico Stage 6"
  "2017 Tour de Pologne Stage 2"
  "2017 Tour de Pologne Stage 3"
  "2017 Tour de Pologne Stage 4"
  "2017 Tour de Pologne Stage 5"
  "2017 Tour de Pologne Stage 6"
  "2017 Tour de Pologne Stage 7"
  "2017 Tour de Romandie Stage 1"
  "2017 Tour de Romandie Stage 3"
  "2017 Tour de Romandie Stage 4"
  "2017 Tour of Oman Stage 5"
  "2017 Tour of Oman Stage 6"
  "

In [ ]:
# manual_add("-fAon4ijUa0", "2017 Tour de Pologne Stage 1")

Manual fetch: 2017 Tour de Pologne Stage 1
  Video ID:  -fAon4ijUa0
  Race:      Tour de Pologne
  Date:      2017-07-29
  Stage:     1
  ✓ 20,439 chars | 22.2 min
  Saved: 2017_tour_de_pologne_stage_1.json


{'success': True, 'chars': 20439}

In [26]:
def retry_ip_blocked(delay_seconds: float = 90.0) -> dict:
    """
    Retry all races where transcript fetch was blocked by YouTube IP limiting.
    No YouTube Data API calls — zero quota cost.
    """
    blocked = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        status   = data.get('status', '')
        video    = data.get('video') or {}
        video_id = video.get('video_id')
        # Match the actual error message YouTube returns
        is_blocked = (
            'blocking requests from your ip' in status.lower() or
            'ip has been blocked' in status.lower() or
            'too many requests' in status.lower() or
            'requestblocked' in status.lower() or
            'ipblocked' in status.lower()
        )

        if is_blocked and video_id:
            blocked.append((path, data))

    print(f"IP blocked races found: {len(blocked)}")
    print(f"Delay between requests: {delay_seconds}s")
    print(f"Estimated time:         {len(blocked) * delay_seconds / 60:.0f} minutes")
    print(f"{'─'*50}")

    success = still_blocked = errors = 0

    for path, data in blocked:
        label    = data.get('label', path.stem)
        video_id = data['video']['video_id']

        print(f"\nRetrying: {label}")
        transcript = fetch_transcript(video_id)

        if transcript['success']:
            data['transcript'] = {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            }
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f"  ✓ {transcript['clean_chars']:,} chars")
            success += 1
        else:
            error    = transcript['error']
            is_still = 'blocking requests from your ip' in error.lower() or \
                       'ip has been blocked' in error.lower()
            if is_still:
                print(f"  ✗ Still IP blocked — stopping")
                still_blocked += 1
                break
            else:
                print(f"  ✗ {error[:80]}")
                data['status'] = f'transcript_error:{error[:100]}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1

        actual_delay = delay_seconds + random.uniform(-10, 15)
        actual_delay = max(30, actual_delay)
        print(f"  Waiting {actual_delay:.0f}s...")
        time.sleep(actual_delay)

    print(f"\n{'─'*50}")
    print(f"Retry complete")
    print(f"  Saved:         {success}")
    print(f"  Still blocked: {still_blocked}")
    print(f"  Other errors:  {errors}")
    return {'success': success, 'still_blocked': still_blocked, 'errors': errors}

In [27]:
retry_ip_blocked()

IP blocked races found: 0
Delay between requests: 90.0s
Estimated time:         0 minutes
──────────────────────────────────────────────────

──────────────────────────────────────────────────
Retry complete
  Saved:         0
  Still blocked: 0
  Other errors:  0


{'success': 0, 'still_blocked': 0, 'errors': 0}

In [23]:
run_transcript_fetch(max_transcripts=20, delay_seconds=60)

Videos pending transcript: 0
Fetching up to:            0
Delay between requests:    60s
Estimated time:            0.0 minutes
———————————————————————————————————————————————————————

———————————————————————————————————————————————————————
Fetch complete
  Transcripts saved:   0
  IP blocked:          0
  Other errors:        0
  Still pending retry: 0


{'success': 0, 'ip_blocked': 0, 'errors': 0}

AUDIT

In [28]:
def audit_transcript_quality(year_filter: int = None) -> None:
    """
    Scan saved transcripts and flag ones that look like
    preview/preview videos rather than race highlights.
    """
    bad_keywords = [
        'top 10 riders to watch', 'preview', 'riders to watch',
        'team time trial preparation', 'guide to', 'how to watch',
        'what to expect', 'ones to watch'
    ]

    flagged = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)

        if data.get('status') != 'transcript_saved':
            continue

        label = data.get('label', '')
        if year_filter and not label.startswith(str(year_filter)):
            continue

        title = data.get('video', {}).get('title', '').lower()
        if any(kw in title for kw in bad_keywords):
            flagged.append({
                'label': label,
                'title': data['video']['title'],
                'path':  path,
            })

    print(f"Flagged as likely bad matches: {len(flagged)}\n")
    for f in flagged:
        print(f"  {f['label']}")
        print(f"    {f['title']}")
        print()

    return flagged

bad_matches = audit_transcript_quality()

Flagged as likely bad matches: 5

  2017 Criterium du Dauphine Stage 1
    10 Riders To Watch At The 2017 Critérium du Dauphiné

  2017 Ronde Van Vlaanderen
    Tour Of Flanders 2017 – GCN&#39;s Top 11 Riders To Watch

  2017 Vuelta a Espana Stage 1
    GCN&#39;s Top 10 Riders To Watch At The Vuelta A España 2017

  2017 Vuelta a Espana Stage 2
    GCN&#39;s Top 10 Riders To Watch At The Vuelta A España 2017

  2017 Vuelta a Espana Stage 3
    Team Time Trial Preparation With BMC Racing Team | Vuelta a España 2017



In [30]:
def delete_bad_matches(flagged: list, dry_run: bool = True) -> None:
    """
    Delete flagged bad match files and reset to no_video_found
    so they don't get used in RAG.
    dry_run=True just shows what would be deleted.
    """
    for f in flagged:
        if dry_run:
            print(f"Would delete: {f['path'].name}")
        else:
            # Replace with no_video_found record
            data = json.load(open(f['path']))
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w') as out:
                json.dump(clean, out, indent=2)
            print(f"Reset: {f['path'].name}")

# First dry run to see what would be affected
# delete_bad_matches(bad_matches, dry_run=True)

# When satisfied, run for real:
delete_bad_matches(bad_matches, dry_run=False)

Reset: 2017_criterium_du_dauphine_stage_1.json
Reset: 2017_ronde_van_vlaanderen.json
Reset: 2017_vuelta_a_espana_stage_1.json
Reset: 2017_vuelta_a_espana_stage_2.json
Reset: 2017_vuelta_a_espana_stage_3.json


---
## Cell 12 — Claude tactical extraction
Reads saved transcripts and extracts structured tactical events.
Only processes transcripts not yet extracted — never re-charges for existing work.
Cost: ~$0.02 per race. Run whenever you want, at your own pace.


In [ ]:
EXTRACTION_PROMPT = """You are analyzing cycling race commentary for {race_name}.

Extract tactically significant information from this transcript.
Focus only on information useful for pre-race analysis of future similar stages.
Ignore music, crowd noise, and generic excitement phrases.

Return a JSON object:
{{
  "race_summary": "2-3 sentence tactical summary of what happened",
  "decisive_moment": "the single most important tactical event",
  "commentary_quality": "rich|moderate|thin",
  "events": [
    {{
      "event_type": "attack|breakaway|chase|team_tactic|rider_observation|weather|terrain",
      "riders": ["LASTNAME Firstname"],
      "teams": ["Team Name"],
      "description": "what happened concisely",
      "tactical_significance": "why this matters for future analysis"
    }}
  ],
  "rider_form_signals": [
    {{
      "rider": "LASTNAME Firstname",
      "signal": "positive|negative|neutral",
      "observation": "what commentary reveals about their condition"
    }}
  ],
  "terrain_observations": [
    "observation about how terrain affected racing dynamics"
  ],
  "usable_for_rag": true
}}

Return only valid JSON, no other text.
Transcript:
{transcript}"""


def extract_one(label: str, transcript_text: str) -> dict:
    max_chars = 8000
    if len(transcript_text) > max_chars:
        half = max_chars // 2
        text = transcript_text[:half] + '\n[...middle omitted...]\n' + transcript_text[-half:]
    else:
        text = transcript_text

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=3000,
        messages=[{'role': 'user', 'content': EXTRACTION_PROMPT.format(
            race_name=label, transcript=text
        )}]
    )
    raw = re.sub(r'```json|```', '', response.content[0].text).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'json_parse_failed', 'raw': raw[:200]}


def run_extraction(max_extractions: int = 50, verbose: bool = True) -> dict:
    """
    Extract tactical events from saved transcripts using Claude.
    Only processes transcripts not yet extracted.
    Never re-processes an already-extracted race.
    """
    pending = [
        path for path in sorted(RAW_DIR.glob('*.json'))
        if not (EXTRACTED_DIR / path.name).exists()
        and json.load(open(path)).get('status') == 'transcript_saved'
    ]

    total_cost = success = skipped = errors = 0
    print(f'Transcripts pending extraction: {len(pending)}')
    print(f'Processing up to: {max_extractions}')
    print(f'Est. cost: ${min(len(pending), max_extractions) * 0.02:.2f}')
    print(f'{chr(8212)*55}')

    for path in pending[:max_extractions]:
        extracted_path = EXTRACTED_DIR / path.name

        with open(path, encoding='utf-8') as f:
            data = json.load(f)

        transcript_text = data.get('transcript', {}).get('clean_text', '')
        if not transcript_text.strip():
            skipped += 1
            continue

        label = data.get('label', path.stem)
        chars = len(transcript_text)
        cost  = min(chars, 8000) / 4 / 1_000_000 * 3

        if verbose:
            print(f'Extracting: {label}')
            print(f'  Chars: {chars:,} | Est: ${cost:.4f}')

        try:
            events = extract_one(label, transcript_text)
            output = {
                'label':      label,
                'race_name':  data.get('race_name'),
                'race_date':  data.get('race_date'),
                'stage':      data.get('stage'),
                'video_title': data.get('video', {}).get('title'),
                'video_url':  data.get('video', {}).get('url'),
                'channel':    data.get('video', {}).get('channel'),
                'extraction': events,
                'extracted_at': str(datetime.datetime.utcnow()),
            }
            with open(extracted_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            total_cost += cost
            success += 1
            if verbose:
                quality  = events.get('commentary_quality', 'unknown')
                n_events = len(events.get('events', []))
                print(f'  ✓ Quality: {quality} | Events: {n_events}')

        except Exception as e:
            print(f'  ✗ Error: {e}')
            errors += 1

        time.sleep(0.5)

    print(f'\n{chr(8212)*55}')
    print(f'Extraction complete')
    print(f'  Extracted:   {success}')
    print(f'  Skipped:     {skipped}')
    print(f'  Errors:      {errors}')
    print(f'  Total cost:  ${total_cost:.4f}')
    print(f'  Total in extracted dir: {len(list(EXTRACTED_DIR.glob("*.json")))}')
    return {'success': success, 'skipped': skipped, 'errors': errors, 'cost': total_cost}


# ── RUN EXTRACTION ─────────────────────────────────────────
# extraction_stats = run_extraction(max_extractions=50, verbose=True)
print('Extraction functions ready')
print('Uncomment the last line to run extraction')


---
## Cell 13 — Commentary retrieval
Used by the agent notebook. Always returns something useful.


In [ ]:
def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 3000
) -> str:
    """
    Returns formatted context string for the LLM.
    Priority: extracted events > raw transcript > no commentary message.
    Never raises — always returns a string.
    """
    label     = make_label(race_name, race_date, stage)
    safe_name = make_safe_name(label)

    extracted_path = EXTRACTED_DIR / f'{safe_name}.json'
    raw_path       = RAW_DIR       / f'{safe_name}.json'

    # Option 1 — extracted tactical events
    if extracted_path.exists():
        try:
            with open(extracted_path, encoding='utf-8') as f:
                data = json.load(f)
            ext = data.get('extraction', {})
            if 'error' not in ext:
                parts = [f"[COMMENTARY: {data.get('channel')} | {data.get('video_title','')[:55]}]"]
                if ext.get('race_summary'):
                    parts.append(f"Summary: {ext['race_summary']}")
                if ext.get('decisive_moment'):
                    parts.append(f"Decisive moment: {ext['decisive_moment']}")
                for e in ext.get('events', [])[:5]:
                    riders = ', '.join(e.get('riders', []))
                    parts.append(f"- [{e['event_type']}] {e['description']}" +
                                 (f' ({riders})' if riders else ''))
                for s in ext.get('rider_form_signals', [])[:3]:
                    parts.append(f"- Form: {s['rider']} ({s['signal']}): {s['observation']}")
                return '\n'.join(parts)
        except Exception:
            pass

    # Option 2 — raw transcript
    if raw_path.exists():
        try:
            with open(raw_path, encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'transcript_saved':
                text = data.get('transcript', {}).get('clean_text', '')
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + '\n[...]\n' + text[-half:]
                    channel = data.get('video', {}).get('channel', 'unknown')
                    title   = data.get('video', {}).get('title', '')[:55]
                    return f'[RAW COMMENTARY: {channel} | {title}]\n\n{text}'
        except Exception:
            pass

    # Option 3 — nothing available
    return f'[NO COMMENTARY for {label}] Analysis based on structured data only.'


# Quick test
print('=== Commentary retrieval test ===')
test_cases = [
    ('Tour de France',  '2023-07-19', 17),
    ('Paris-Roubaix',   '2023-04-09', None),
    ("Giro d'Italia",   '2022-05-15', 9),
]
for race, date, stage in test_cases:
    ctx = get_commentary_context(race, date, stage, max_chars=200)
    print(f'\n{date[:4]} {race}' + (f' Stage {stage}' if stage else ''))
    print(ctx[:150])
